In [4]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [3]:
TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\train"
VALID_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\valid"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\test"

In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
FINE_TUNE_AT = 15
LR_HEAD = 1e-3


In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)



In [6]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)


valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 11132 images belonging to 12 classes.
Found 240 images belonging to 12 classes.
Found 600 images belonging to 12 classes.


In [7]:
num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print("Classes:", num_classes)
print(class_names)


Classes: 12
['Black Rust', 'Blast', 'Brown Rust', 'Common Root Rot', 'Fusarium Head Blight', 'Healthy', 'Leaf Blight', 'Mildew', 'Septoria', 'Smut', 'Tan spot', 'Yellow Rust']


In [8]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False  # stage 1

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 12)                  │           3,084 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,849,100 (56.64 MB)

 Trainable params: 134,412 (525.05 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [9]:

def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    return loss

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)
history_head = model.fit(
    train_generator,
    epochs=EPOCHS_HEAD,
    validation_data=valid_generator,
    verbose=1
)

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 985s 3s/step - accuracy: 0.1641 - loss: 1.3223 - val_accuracy: 0.3167 - val_loss: 0.6151
Epoch 2/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 929s 3s/step - accuracy: 0.3545 - loss: 0.4659 - val_accuracy: 0.4208 - val_loss: 0.4709
Epoch 3/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 788s 2s/step - accuracy: 0.4322 - loss: 0.3480 - val_accuracy: 0.4833 - val_loss: 0.4215
Epoch 4/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 876s 3s/step - accuracy: 0.4934 - loss: 0.2954 - val_accuracy: 0.4875 - val_loss: 0.4030
Epoch 5/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 3551s 10s/step - accuracy: 0.5097 - loss: 0.2803 - val_accuracy: 0.5125 - val_loss: 0.3806
Epoch 6/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 1080s 3s/step - accuracy: 0.5415 - loss: 0.2547 - val_accuracy: 0.5417 - val_loss: 0.3628
Epoch 7/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 4994s 14s/step - accuracy: 0.5716 - loss: 0.2356 - val_accuracy: 0.5833 - val_loss: 0.3760
Epoch 8/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 955s 3s/step - accuracy: 0.5899 - loss: 0.2242 - val

In [10]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_generator,
    epochs=EPOCHS_FINE,
    validation_data=valid_generator,
    verbose=1
)

Epoch 1/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 67610s 195s/step - accuracy: 0.6457 - loss: 1.0721 - val_accuracy: 0.6333 - val_loss: 2.2636
Epoch 2/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 933s 3s/step - accuracy: 0.6880 - loss: 0.9237 - val_accuracy: 0.6583 - val_loss: 2.0321
Epoch 3/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 921s 3s/step - accuracy: 0.7231 - loss: 0.8124 - val_accuracy: 0.7042 - val_loss: 2.0858
Epoch 4/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 986s 3s/step - accuracy: 0.7448 - loss: 0.7530 - val_accuracy: 0.7333 - val_loss: 1.8571
Epoch 5/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 847s 2s/step - accuracy: 0.7670 - loss: 0.6898 - val_accuracy: 0.7500 - val_loss: 2.1194
Epoch 6/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 982s 3s/step - accuracy: 0.7753 - loss: 0.6617 - val_accuracy: 0.7500 - val_loss: 2.1330
Epoch 7/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 826s 2s/step - accuracy: 0.7923 - loss: 0.6027 - val_accuracy: 0.7625 - val_loss: 2.2450
Epoch 8/20
348/348 ━━━━━━━━━━━━━━━━━━━━ 779s 2s/step - accuracy: 0.8058 - loss: 0.5663 - val_

In [11]:
model.save('VGG16_WPD.keras')

In [12]:
with open("training_history_VGG16_2.json", "w", encoding="utf-8") as f:
    json.dump(history_head.history, f, indent=2)

print("Saved: training_history_VGG16_2.json")

Saved: training_history_VGG16_2.json


In [13]:
with open("training_history_ft_VGG16_2.json", "w", encoding="utf-8") as f:
    json.dump(history_ft.history, f, indent=2)

print("Saved: training_history_ft_VGG16_2.json")

Saved: training_history_ft_VGG16_2.json


In [14]:
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

19/19 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step - accuracy: 0.8429 - loss: 2.1634
Test loss: 2.5701
Test accuracy: 0.8500


In [15]:
test_metrics = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy)
}

with open("test_metric_VGG16_2.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)


Saved test_metrics_VGG16_2.json


In [16]:
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

19/19 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step


In [17]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


                           precision    recall  f1-score   support

          black_rust_test       0.97      0.72      0.83        50
               blast_test       0.89      1.00      0.94        50
          brown_rust_test       0.78      0.98      0.87        50
     common_root_rot_test       0.98      0.94      0.96        50
fusarium_head_blight_test       1.00      0.90      0.95        50
             healthy_test       0.67      0.08      0.14        50
         leaf_blight_test       0.79      0.90      0.84        50
              mildew_test       0.96      0.94      0.95        50
            septoria_test       0.98      0.94      0.96        50
                smut_test       0.96      1.00      0.98        50
            tan_spot_test       0.89      0.80      0.84        50
         yellow_rust_test       0.53      1.00      0.69        50

                 accuracy                           0.85       600
                macro avg       0.87      0.85      0.83    

In [24]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=class_names,      
    columns=class_names     
)

print(cm_df)


                           black_rust_test  blast_test  brown_rust_test  \
black_rust_test                         38           0               12   
blast_test                               0          50                0   
brown_rust_test                          0           0               47   
common_root_rot_test                     0           0                0   
fusarium_head_blight_test                0           3                0   
healthy_test                             0           2                0   
leaf_blight_test                         1           0                0   
mildew_test                              1           0                0   
septoria_test                            0           0                0   
smut_test                                0           0                0   
tan_spot_test                            0           0                1   
yellow_rust_test                         0           0                0   

                        

In [25]:
cm_df

,black_rust_test,blast_test,brown_rust_test,common_root_rot_test,fusarium_head_blight_test,healthy_test,leaf_blight_test,mildew_test,septoria_test,smut_test,tan_spot_test,yellow_rust_test
black_rust_test,38,0,12,0,0,0,0,0,0,0,0,0
blast_test,0,50,0,0,0,0,0,0,0,0,0,0
brown_rust_test,0,0,47,0,0,3,0,0,0,0,0,0
common_root_rot_test,0,0,0,50,0,0,0,0,0,0,0,0
fusarium_head_blight_test,0,3,0,0,45,1,0,0,0,0,1,0
healthy_test,0,2,0,0,0,4,1,0,0,0,0,43
leaf_blight_test,1,0,0,0,0,1,44,0,0,0,4,0
mildew_test,1,0,0,1,0,0,1,47,0,0,0,0
septoria_test,0,0,0,0,0,0,2,0,48,0,0,0
smut_test,0,0,0,1,0,0,0,0,0,49,0,0
